# Expériences de sélection de méthode d'entropie et seuil d'alerte robuste

Structure du notebook :
1. **Partie 0** – Imports et configuration
2. **Partie 1** – Score d'entropie hybride (numérique + sémantique) avec tuning du paramètre α
3. **Partie 2** – Plan d'expériences complet (Test 1 : données chiffrées, Test 2 : données textuelles)
4. **Partie 3** – Tableau de résultats et choix final
5. **Partie 4** – Seuil d'alerte robuste par bootstrap

---
## Partie 0 — Imports et configuration

In [ ]:
import re
import time
import json
import yaml
import random
import urllib3
import os
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats import spearmanr
from scipy.special import expit  # sigmoid
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Reproductibilité globale ──────────────────────────────────────────────────
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

print("Imports OK")

In [ ]:
from datasets.dataset_monitoring import Dataset
from modules.semantic_entropy import SemanticEntropyModule
from core.pipeline import MonitoringPipeline
import pandas as pd

# Chargement de ton DataFrame pré-généré.
# Colonnes attendues :
#   id_prompt          : str   — identifiant de la tâche (ex. "g_0", "clean_g_3")
#   responses          : list[str]  — toutes les réponses disponibles pour ce prompt
#   label              : int   — 0 = propre, 1 = perturbé léger, 2 = très perturbé
#   n1                 : int   — nombre de réponses de niveau 1 dans le lot
#   n2                 : int   — nombre de réponses de niveau 2 dans le lot
#   N                  : int   — taille totale du lot
#   score_perturbation : float — (n1 + 2*n2) / N  (déjà calculé)
#
# Adapte le chemin :
all_data_df = pd.read_pickle("../data/all_responses.pkl")
# ou : all_data_df = pd.read_json("../data/all_responses.json")

# Vérification
assert 'score_perturbation' in all_data_df.columns, \
    "Colonne score_perturbation manquante — calcule-la avec : (n1 + 2*n2) / N"

print(f"Données chargées : {len(all_data_df)} prompts")
print(f"Distribution labels :\n{all_data_df['label'].value_counts().sort_index()}")
print(f"Score perturbation — min={all_data_df['score_perturbation'].min():.3f} "
      f"max={all_data_df['score_perturbation'].max():.3f} "
      f"moy={all_data_df['score_perturbation'].mean():.3f}")


In [ ]:
# ── Hyperparamètres à balayer ─────────────────────────────────────────────────
BATCH_SIZES      = [10, 20, 40]
CLUSTERING_METHODS = ["threshold", "dbscan", "gmm", "kmeans", "agglomerative"]
N_LOTS           = 20        # lots par tâche
N_BOOTSTRAP      = 1000      # itérations bootstrap pour le seuil final
ALPHA_GRID       = np.linspace(0, 1, 21)  # de 0.0 à 1.0 par pas de 0.05

# Modèles d'embedding
EMBED_MODEL_TEXT    = "paraphrase-multilingual-mpnet-base-v2"   # textuel
EMBED_MODEL_FINANCE = "paraphrase-multilingual-mpnet-base-v2"   # à remplacer par FinBERT si dispo

print(f"Grille d'expériences : {len(BATCH_SIZES)} tailles × {len(CLUSTERING_METHODS)} méthodes"
      f" = {len(BATCH_SIZES)*len(CLUSTERING_METHODS)} combinaisons par test")

---
## Partie 1 — Score d'entropie hybride et tuning du paramètre α

**Problème** : Les embeddings généralistes ne distinguent pas `15.25 %` de `15.45 %`.  
**Solution** : Combiner l'entropie sémantique (clustering embedding) avec une entropie numérique  
qui mesure la dispersion des valeurs chiffrées extraites par regex.

$$H_{\text{hybride}} = (1-\alpha) \cdot H_{\text{sémantique}} + \alpha \cdot H_{\text{numérique}}$$

α = 0 → entropie purement sémantique  
α = 1 → entropie purement numérique  
Le bon α est celui qui maximise la corrélation de Spearman avec le score de perturbation.

In [ ]:
# ── 1.1  Extraction et entropie des valeurs numériques ───────────────────────

def extract_numbers(text: str) -> list[float]:
    """Extrait tous les nombres d'un texte (entiers, décimaux, négatifs)."""
    # Gère : -5.80, 21,4, 15.25%, 33.45
    pattern = r'-?\d+[.,]?\d*'
    matches = re.findall(pattern, text.replace(',', '.'))
    return [float(m) for m in matches if m not in ('.', '-', '')]


def numeric_entropy(responses: list[str], eps: float = 1e-9) -> float:
    """
    Entropie numérique d'un lot de réponses.

    Principe :
    - Pour chaque position de nombre dans la réponse de référence, on collecte
      les valeurs correspondantes dans toutes les réponses.
    - On calcule le coefficient de variation (std/mean) pour chaque position.
    - L'entropie numérique est la moyenne de ces CV, normalisée en [0,1] via
      une sigmoïde.

    Si aucun nombre n'est détecté, retourne 0.
    """
    if not responses:
        return 0.0

    all_numbers = [extract_numbers(r) for r in responses]
    # On garde uniquement les positions présentes dans toutes les réponses
    min_len = min((len(n) for n in all_numbers), default=0)
    if min_len == 0:
        return 0.0

    cvs = []
    for pos in range(min_len):
        vals = np.array([n[pos] for n in all_numbers])
        mean_abs = np.abs(np.mean(vals))
        if mean_abs < eps:
            # Si la valeur moyenne est ~0, on utilise l'écart-type absolu
            cvs.append(np.std(vals))
        else:
            cvs.append(np.std(vals) / mean_abs)

    if not cvs:
        return 0.0

    mean_cv = np.mean(cvs)
    # Normalisation sigmoïde centrée : CV=0 → 0, CV=0.05 → ~0.5 (seuil finance)
    # Facteur k=40 : sensible aux variations de l'ordre du pourcent
    k = 40
    normalized = 2 * expit(k * mean_cv) - 1  # dans [0, 1]
    return float(np.clip(normalized, 0.0, 1.0))


# ── Test unitaire rapide ──────────────────────────────────────────────────────
r_identiques = ["Le taux est de 15.25% en 2023."] * 5
r_variables  = ["Le taux est de 15.25% en 2023.",
                "Le taux est de 15.45% en 2023.",
                "Le taux est de 15.10% en 2023.",
                "Le taux est de 15.70% en 2023.",
                "Le taux est de 14.90% en 2023."]

print(f"Entropie numérique réponses IDENTIQUES  : {numeric_entropy(r_identiques):.4f}  (attendu ≈ 0)")
print(f"Entropie numérique réponses VARIABLES   : {numeric_entropy(r_variables):.4f}   (attendu > 0)")

In [ ]:
# ── 1.2  Score hybride ────────────────────────────────────────────────────────

def hybrid_entropy(
    semantic_entropy: float,
    responses: list[str],
    alpha: float
) -> float:
    """
    H_hybrid = (1-α) * H_sémantique + α * H_numérique

    Args:
        semantic_entropy : entropie déjà calculée par SemanticEntropyModule
        responses        : liste brute des réponses du lot
        alpha            : poids de la composante numérique (dans [0,1])

    Returns:
        score hybride dans [0, +∞) (l'entropie sémantique peut dépasser 1)
    """
    h_num = numeric_entropy(responses)
    return (1 - alpha) * semantic_entropy + alpha * h_num


print("Fonction hybrid_entropy définie.")

In [ ]:
# ── 1.3  Tuning de α sur les données chiffrées ───────────────────────────────
#
# On suppose que tu as déjà un DataFrame `all_lots_df_finance` issu de ton
# pipeline existant sur les données finance, avec au moins les colonnes :
#   - responses        : list[str]  (les N réponses du lot)
#   - score_perturbation : float    (0.0 = propre, 0.7 = très perturbé)
#   - entropy          : float      (entropie sémantique déjà calculée)
#
# Si tu n'as pas encore ce DataFrame, exécute d'abord les cellules de génération
# de lots de la Partie 2 avec la méthode de clustering de référence (dbscan,
# batch_size=20) et enregistre les résultats ici.
#
# ─────────────────────────────────────────────────────────────────────────────
# PLACEHOLDER : remplace par ton vrai chargement
# all_lots_df_finance = pd.read_pickle("lots_finance_reference.pkl")
# ─────────────────────────────────────────────────────────────────────────────

def tune_alpha(
    df: pd.DataFrame,
    alpha_grid: np.ndarray = ALPHA_GRID,
    responses_col: str = "responses",
    semantic_col: str  = "entropy",
    perturb_col:  str  = "score_perturbation",
) -> tuple[float, pd.DataFrame]:
    """
    Balaye les valeurs de α et renvoie le α optimal (max corrélation Spearman)
    ainsi qu'un DataFrame des résultats pour visualisation.
    """
    records = []
    for alpha in alpha_grid:
        hybrid_scores = df.apply(
            lambda row: hybrid_entropy(
                row[semantic_col],
                row[responses_col],
                alpha
            ),
            axis=1
        )
        rho, pval = spearmanr(df[perturb_col], hybrid_scores)
        records.append({"alpha": alpha, "spearman_rho": rho, "p_value": pval})

    results_df = pd.DataFrame(records)
    best_row   = results_df.loc[results_df["spearman_rho"].idxmax()]
    best_alpha = float(best_row["alpha"])

    return best_alpha, results_df


def plot_alpha_tuning(results_df: pd.DataFrame, best_alpha: float, title: str = ""):
    """Courbe de corrélation Spearman en fonction de α."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(results_df["alpha"], results_df["spearman_rho"],
            marker='o', markersize=4, linewidth=1.8, color='steelblue')
    ax.axvline(best_alpha, color='crimson', linestyle='--', linewidth=1.5,
               label=f"α* = {best_alpha:.2f}")
    best_rho = results_df.loc[results_df["alpha"] == best_alpha, "spearman_rho"].values[0]
    ax.axhline(best_rho, color='crimson', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_xlabel("α  (poids entropie numérique)", fontsize=11)
    ax.set_ylabel("Corrélation de Spearman (ρ)", fontsize=11)
    ax.set_title(title or "Tuning du paramètre α", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
    print(f"→ α optimal : {best_alpha:.2f}  |  ρ_max = {best_rho:.4f}")


# ── Décommenter et adapter quand all_lots_df_finance est disponible ───────────
# best_alpha_finance, alpha_results_finance = tune_alpha(all_lots_df_finance)
# plot_alpha_tuning(alpha_results_finance, best_alpha_finance,
#                  title="Tuning α — données chiffrées (finance)")

print("Fonctions de tuning α définies — à appeler après génération des lots (Partie 2).")

---
## Partie 2 — Plan d'expériences complet

**Règle clé** : pour une taille de lot donnée, les **mêmes lots** (même seed) sont utilisés  
pour toutes les expériences (méthodes de clustering différentes).  
Cela garantit que la corrélation mesure uniquement l'effet de la méthode.

In [ ]:
# ── 2.1  Sampling des lots à partir des données pré-générées ──────────────────
#
# Pour chaque id_prompt, on tire N_LOTS lots de taille batch_size
# en échantillonnant SANS remise parmi les réponses existantes.
#
# RÈGLE CLÉ : seed fixe → mêmes lots pour toutes les méthodes de clustering
# sur une taille de lot donnée.
#
# score_perturbation du lot = (n1 + 2*n2) / N
# avec n1 = réponses de label 1, n2 = réponses de label 2 dans le lot tiré.

def build_lots_from_df(
    df: pd.DataFrame,
    batch_size: int,
    n_lots: int = N_LOTS,
    seed: int = GLOBAL_SEED,
    responses_col: str = "responses",
    label_col: str = "label",
) -> list[dict]:
    """
    Génère n_lots lots de taille batch_size pour chaque id_prompt.

    Chaque lot est un dict :
        {
          'task_id'            : str,    # id_prompt
          'lot_id'             : int,
          'responses'          : list[str],
          'score_perturbation' : float,  # (n1 + 2*n2) / N
        }

    Les réponses disponibles par prompt sont sous forme de list[str]
    dans la colonne `responses_col`.
    Si le prompt a moins de réponses que batch_size, on tire AVEC remise.
    """
    rng  = np.random.default_rng(seed)
    lots = []

    for _, row in df.iterrows():
        task_id   = row["id_prompt"]
        responses = row[responses_col]           # list[str]
        labels    = row.get(label_col + "s",     # list[int] parallèle aux réponses
                            [row[label_col]] * len(responses))  # fallback si une seule valeur

        n_available = len(responses)
        replace     = n_available < batch_size   # tirage avec remise si pool trop petit

        for lot_id in range(n_lots):
            idx    = rng.choice(n_available, size=batch_size, replace=replace)
            sampled_responses = [responses[i] for i in idx]
            sampled_labels    = [labels[i]    for i in idx]

            # Score de perturbation du lot = (n1 + 2*n2) / N
            n1 = sampled_labels.count(1)
            n2 = sampled_labels.count(2)
            score = (n1 + 2 * n2) / batch_size

            lots.append({
                "task_id"            : task_id,
                "lot_id"             : lot_id,
                "responses"          : sampled_responses,
                "score_perturbation" : score,
            })

    print(f"[build_lots] batch_size={batch_size} → {len(lots)} lots "
          f"({len(df)} tâches × {n_lots} lots) | "
          f"score moyen={np.mean([l['score_perturbation'] for l in lots]):.3f}")
    return lots


# Génération des lots pour les 3 tailles (une seule fois, partagés entre méthodes)
# ── Test 1 : données finance ──────────────────────────────────────────────────
df_finance = all_data_df[all_data_df["test"] == "finance"].reset_index(drop=True)
# ── Test 2 : données textuelles ───────────────────────────────────────────────
df_text    = all_data_df[all_data_df["test"] == "text"].reset_index(drop=True)
# Adapte le filtre selon le nom de ta colonne de distinction des deux tests.

lots_finance = {bs: build_lots_from_df(df_finance, batch_size=bs) for bs in BATCH_SIZES}
lots_text    = {bs: build_lots_from_df(df_text,    batch_size=bs) for bs in BATCH_SIZES}


In [ ]:
# ── 2.2  Calcul d'entropie pour un lot et une méthode de clustering ───────────

def compute_entropy_for_lot(
    responses: list[str],
    clustering_method: str,
    embed_model_name: str,
    alpha: float = 0.0,  # 0 = purement sémantique ; > 0 = hybride
    # Hyperparamètres passés à SemanticEntropyModule
    similarity_threshold: float = 0.9,
    n_clusters: int | None = None,
) -> float:
    """
    Calcule H_hybride pour un lot de réponses avec la méthode de clustering donnée.

    Pour 'kmeans' et 'agglomerative', n_clusters est estimé automatiquement
    via le score silhouette si non fourni.
    """
    dataset = Dataset()
    dataset.add_sample(prompt="lot", response={i: r for i, r in enumerate(responses)})

    # Mapping nom → paramètres SemanticEntropyModule
    method_kwargs = {
        "threshold"    : dict(clustering_method="threshold",
                              similarity_threshold=similarity_threshold),
        "dbscan"       : dict(clustering_method="dbscan"),
        "gmm"          : dict(clustering_method="gmm"),
        "kmeans"       : dict(clustering_method="kmeans",
                              n_clusters=n_clusters),
        "agglomerative": dict(clustering_method="agglomerative"),
    }

    if clustering_method not in method_kwargs:
        raise ValueError(f"Méthode inconnue : {clustering_method}. "
                         f"Choisir parmi {list(method_kwargs.keys())}")

    module = SemanticEntropyModule(
        method="embedding_clustering",
        name="exp",
        model_name=embed_model_name,
        **method_kwargs[clustering_method]
    )

    pipeline = MonitoringPipeline(
        dataset=dataset,
        modules=[module],
        use_existing_responses=True,
    )
    results = pipeline.run()

    semantic_ent = results[0]['modules']['exp']['entropy'] if results else 0.0
    return hybrid_entropy(semantic_ent, responses, alpha)


print("Fonction compute_entropy_for_lot définie.")

In [ ]:
# ── 2.3  Boucle principale sur toutes les combinaisons d'hyperparamètres ──────

def run_experiment(
    lots_by_batch_size: dict[int, list[dict]],
    clustering_methods: list[str],
    embed_model_name:  str,
    alpha:             float = 0.0,
    test_name:         str  = "test",
) -> pd.DataFrame:
    """
    Exécute toutes les combinaisons (batch_size × clustering_method).

    Args:
        lots_by_batch_size : dict { batch_size: [lot_dict, ...] }
                             Les lots sont DÉJÀ générés et partagés entre méthodes.
        clustering_methods : liste des méthodes à tester
        embed_model_name   : modèle d'embedding
        alpha              : paramètre hybride (0 = sémantique pur)
        test_name          : étiquette du test pour le tableau final

    Returns:
        DataFrame avec colonnes :
        test | batch_size | clustering | spearman_rho | p_value | temps_moyen_s
    """
    rows = []
    total = len(lots_by_batch_size) * len(clustering_methods)
    done  = 0

    for batch_size, lots in lots_by_batch_size.items():
        for method in clustering_methods:
            done += 1
            print(f"[{done}/{total}] {test_name} | batch={batch_size} | méthode={method} …",
                  end=" ", flush=True)

            entropies    = []
            perturbations = []
            times        = []

            for lot in lots:
                t0 = time.perf_counter()
                h  = compute_entropy_for_lot(
                        responses=lot['responses'],
                        clustering_method=method,
                        embed_model_name=embed_model_name,
                        alpha=alpha,
                     )
                elapsed = time.perf_counter() - t0
                entropies.append(h)
                perturbations.append(lot['score_perturbation'])
                times.append(elapsed)

            rho, pval = spearmanr(perturbations, entropies)
            print(f"ρ = {rho:.3f}  (p={pval:.3f})  |  t_moy = {np.mean(times):.2f}s")

            rows.append({
                "test"          : test_name,
                "batch_size"    : batch_size,
                "clustering"    : method,
                "spearman_rho"  : round(rho,  4),
                "p_value"       : round(pval, 4),
                "temps_moyen_s" : round(np.mean(times), 3),
            })

    return pd.DataFrame(rows)


print("Fonction run_experiment définie.")

In [ ]:
# ── TEST 1 : données chiffrées (finance) ─────────────────────────────────────
# Les lots sont déjà dans lots_finance (généré en cellule 2.1).
# On calcule les entropies de référence (dbscan, batch=20) pour le tuning α.

ref_lots_finance = lots_finance[20]

ref_records = []
for lot in ref_lots_finance:
    sem_ent = compute_entropy_for_lot(
        responses=lot['responses'],
        clustering_method="dbscan",
        embed_model_name=EMBED_MODEL_FINANCE,
        alpha=0.0,
    )
    ref_records.append({
        'responses'          : lot['responses'],
        'entropy'            : sem_ent,
        'score_perturbation' : lot['score_perturbation'],
    })

ref_df_finance = pd.DataFrame(ref_records)
print(f"Référence finance : {len(ref_df_finance)} lots")
print(f"Corrélation Spearman de base (α=0) : "
      f"{spearmanr(ref_df_finance['score_perturbation'], ref_df_finance['entropy'])[0]:.4f}")


In [ ]:
# 3) Tuning α sur les lots de référence (batch=20, méthode=dbscan)
#    On construit un DataFrame minimal avec les entropies sémantiques de référence

ref_lots_finance = lots_finance[20]   # lots de référence

ref_records = []
for lot in ref_lots_finance:
    sem_ent = compute_entropy_for_lot(
        responses=lot['responses'],
        clustering_method="dbscan",
        embed_model_name=EMBED_MODEL_FINANCE,
        alpha=0.0,
    )
    ref_records.append({
        'responses'          : lot['responses'],
        'entropy'            : sem_ent,
        'score_perturbation' : lot['score_perturbation'],
    })

ref_df_finance = pd.DataFrame(ref_records)

best_alpha_finance, alpha_results_finance = tune_alpha(ref_df_finance)
plot_alpha_tuning(
    alpha_results_finance,
    best_alpha_finance,
    title="Tuning α — Test 1 (données chiffrées)"
)

print(f"\n→ α retenu pour Test 1 : {best_alpha_finance:.2f}")

In [ ]:
# 4) Lancement du plan d'expériences Test 1
results_test1 = run_experiment(
    lots_by_batch_size=lots_finance,
    clustering_methods=CLUSTERING_METHODS,
    embed_model_name=EMBED_MODEL_FINANCE,
    alpha=best_alpha_finance,
    test_name="Test1_finance",
)

print("\n=== Résultats Test 1 (données chiffrées) ===")
print(results_test1.sort_values('spearman_rho', ascending=False).to_string(index=False))

In [ ]:
# ── TEST 2 : données textuelles (question-réponse) ───────────────────────────
# Les lots sont déjà dans lots_text (généré en cellule 2.1).

ref_lots_text = lots_text[20]

ref_records_text = []
for lot in ref_lots_text:
    sem_ent = compute_entropy_for_lot(
        responses=lot['responses'],
        clustering_method="dbscan",
        embed_model_name=EMBED_MODEL_TEXT,
        alpha=0.0,
    )
    ref_records_text.append({
        'responses'          : lot['responses'],
        'entropy'            : sem_ent,
        'score_perturbation' : lot['score_perturbation'],
    })

ref_df_text = pd.DataFrame(ref_records_text)
print(f"Référence textuel : {len(ref_df_text)} lots")
print(f"Corrélation Spearman de base (α=0) : "
      f"{spearmanr(ref_df_text['score_perturbation'], ref_df_text['entropy'])[0]:.4f}")


In [ ]:
# Tuning α Test 2 (attendu : α ≈ 0 car peu de chiffres dans les Q&A textuelles)
ref_lots_text = lots_text[20]

ref_records_text = []
for lot in ref_lots_text:
    sem_ent = compute_entropy_for_lot(
        responses=lot['responses'],
        clustering_method="dbscan",
        embed_model_name=EMBED_MODEL_TEXT,
        alpha=0.0,
    )
    ref_records_text.append({
        'responses'          : lot['responses'],
        'entropy'            : sem_ent,
        'score_perturbation' : lot['score_perturbation'],
    })

ref_df_text = pd.DataFrame(ref_records_text)

best_alpha_text, alpha_results_text = tune_alpha(ref_df_text)
plot_alpha_tuning(
    alpha_results_text,
    best_alpha_text,
    title="Tuning α — Test 2 (données textuelles)"
)

print(f"\n→ α retenu pour Test 2 : {best_alpha_text:.2f}")

In [ ]:
# Lancement Test 2
results_test2 = run_experiment(
    lots_by_batch_size=lots_text,
    clustering_methods=CLUSTERING_METHODS,
    embed_model_name=EMBED_MODEL_TEXT,
    alpha=best_alpha_text,
    test_name="Test2_textuel",
)

print("\n=== Résultats Test 2 (données textuelles) ===")
print(results_test2.sort_values('spearman_rho', ascending=False).to_string(index=False))

---
## Partie 3 — Tableau de résultats global et choix final

In [ ]:
# ── 3.1  Fusion et affichage du tableau complet ───────────────────────────────

all_results = pd.concat([results_test1, results_test2], ignore_index=True)

# Pivot pour une lecture côte-à-côte des deux tests
pivot = all_results.pivot_table(
    index=["batch_size", "clustering"],
    columns="test",
    values=["spearman_rho", "temps_moyen_s"]
).round(4)
pivot.columns = ["_".join(c) for c in pivot.columns]
pivot = pivot.reset_index().sort_values(
    "spearman_rho_Test1_finance", ascending=False
)

print("=" * 90)
print("TABLEAU FINAL — Corrélation Spearman (perturbation ↔ entropie) et temps de calcul")
print("=" * 90)
print(pivot.to_string(index=False))

# Sauvegarde CSV
pivot.to_csv("resultats_hyperparams.csv", index=False)
print("\nTableau sauvegardé → resultats_hyperparams.csv")

In [ ]:
# ── 3.2  Heatmaps de corrélation ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (test_name, grp) in zip(axes, all_results.groupby('test')):
    matrix = grp.pivot(index='clustering', columns='batch_size', values='spearman_rho')
    sns.heatmap(
        matrix,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        cbar_kws={"label": "Spearman ρ"}
    )
    ax.set_title(f"Corrélation Spearman\n{test_name}", fontsize=12)
    ax.set_xlabel("Taille du lot (batch_size)", fontsize=10)
    ax.set_ylabel("Méthode de clustering", fontsize=10)

plt.tight_layout()
plt.savefig("heatmap_correlations.png", dpi=150)
plt.show()
print("Figure sauvegardée → heatmap_correlations.png")

In [ ]:
# ── 3.3  Sélection automatique du meilleur couple (batch_size, clustering) ────
#
# Critère : max(ρ_test1 + ρ_test2) avec contrainte temps_moyen < seuil

TEMPS_MAX_S = 60  # seuil de temps de calcul acceptable (à adapter)

# Score composite = moyenne des deux ρ
pivot["score_composite"] = (
    pivot.get("spearman_rho_Test1_finance", 0) +
    pivot.get("spearman_rho_Test2_textuel", 0)
) / 2

# Filtrage par temps
mask_temps = (
    pivot.get("temps_moyen_s_Test1_finance", 0) +
    pivot.get("temps_moyen_s_Test2_textuel", 0)
) / 2 < TEMPS_MAX_S

best_config = pivot[mask_temps].sort_values("score_composite", ascending=False).iloc[0]

BEST_BATCH      = int(best_config["batch_size"])
BEST_CLUSTERING = best_config["clustering"]

print("\n" + "═" * 60)
print(f"  MEILLEURE CONFIGURATION RETENUE")
print(f"  batch_size  : {BEST_BATCH}")
print(f"  clustering  : {BEST_CLUSTERING}")
print(f"  score moyen : {best_config['score_composite']:.4f}")
print("═" * 60)

---
## Partie 4 — Seuil d'alerte robuste par Bootstrap

**Objectif** : remplacer le seuil ponctuel (Youden sur un seul split train/test)  
par un seuil avec intervalle de confiance à 95 %, beaucoup plus défendable en production.

**Principe** :
1. Sur chaque rééchantillonnage bootstrap, on entraîne une régression logistique
   et on extrait le seuil Youden.
2. On collecte les 1000 seuils → distribution empirique.
3. Le seuil final est la **médiane**, les bornes sont les percentiles 2.5 % et 97.5 %.

In [ ]:
# ── 4.1  Construction du DataFrame final avec la meilleure config ─────────────
# score_perturbation = (n1 + 2*n2) / N  →  label_lot = 1 si score > 0

def build_final_df(
    lots: list[dict],
    clustering_method: str,
    embed_model_name: str,
    alpha: float,
) -> pd.DataFrame:
    """Calcule l'entropie hybride pour tous les lots et retourne le DataFrame."""
    records = []
    for lot in lots:
        h = compute_entropy_for_lot(
            responses=lot['responses'],
            clustering_method=clustering_method,
            embed_model_name=embed_model_name,
            alpha=alpha,
        )
        records.append({
            'task_id'            : lot['task_id'],
            'lot_id'             : lot['lot_id'],
            'entropy'            : h,
            'score_perturbation' : lot['score_perturbation'],
            # label binaire pour la logistique et la ROC
            # score > 0 signifie qu'au moins une réponse perturbée est dans le lot
            'label_lot'          : int(lot['score_perturbation'] > 0),
        })
    return pd.DataFrame(records)


final_df_finance = build_final_df(
    lots=lots_finance[BEST_BATCH],
    clustering_method=BEST_CLUSTERING,
    embed_model_name=EMBED_MODEL_FINANCE,
    alpha=best_alpha_finance,
)

final_df_text = build_final_df(
    lots=lots_text[BEST_BATCH],
    clustering_method=BEST_CLUSTERING,
    embed_model_name=EMBED_MODEL_TEXT,
    alpha=best_alpha_text,
)

all_lots_df = pd.concat([final_df_finance, final_df_text], ignore_index=True)
all_lots_df = all_lots_df.sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)

print(f"DataFrame final : {len(all_lots_df)} lots")
print(f"  Score perturbation — min={all_lots_df['score_perturbation'].min():.3f} "
      f"max={all_lots_df['score_perturbation'].max():.3f}")
print(f"  Label positif (score>0) : {all_lots_df['label_lot'].mean()*100:.1f}%")


In [ ]:
# ── 4.2  Bootstrap du seuil d'entropie ───────────────────────────────────────

def youden_entropy_threshold(X: np.ndarray, y: np.ndarray) -> float | None:
    """
    Entraîne une régression logistique sur (entropy → label),
    trouve le seuil de probabilité optimal par l'index de Youden,
    puis le convertit en seuil d'entropie via l'inverse de la sigmoïde.

    Retourne None si la régression échoue (ex. une seule classe dans le fold).
    """
    if len(np.unique(y)) < 2:
        return None

    clf = LogisticRegression(max_iter=500)
    clf.fit(X.reshape(-1, 1), y)

    y_proba = clf.predict_proba(X.reshape(-1, 1))[:, 1]
    fpr, tpr, thresholds = roc_curve(y, y_proba)

    best_idx   = np.argmax(tpr - fpr)
    best_proba = thresholds[best_idx]

    # Seuil proba → seuil entropie
    w, b = clf.coef_[0][0], clf.intercept_[0]
    if abs(w) < 1e-10:
        return None

    z = np.log(best_proba / (1 - best_proba + 1e-12))
    return (z - b) / w


def bootstrap_threshold(
    df: pd.DataFrame,
    entropy_col:  str   = "entropy",
    label_col:    str   = "label_lot",
    n_bootstrap:  int   = N_BOOTSTRAP,
    ci:           float = 0.95,
    seed:         int   = GLOBAL_SEED,
) -> dict:
    """
    Bootstrap du seuil d'entropie.

    Returns:
        {
          'threshold_median' : seuil à utiliser en production,
          'ci_lower'         : borne inférieure de l'IC,
          'ci_upper'         : borne supérieure de l'IC,
          'thresholds'       : liste de tous les seuils bootstrap (pour plot),
          'auc_median'       : AUC médiane sur les rééchantillonnages,
        }
    """
    rng = np.random.default_rng(seed)
    n   = len(df)

    X = df[entropy_col].values
    y = df[label_col].values

    thresholds = []
    aucs       = []

    for i in range(n_bootstrap):
        idx  = rng.integers(0, n, size=n)     # tirage avec remise
        X_b  = X[idx]
        y_b  = y[idx]

        thr = youden_entropy_threshold(X_b, y_b)
        if thr is None:
            continue
        thresholds.append(thr)

        # AUC sur l'échantillon bootstrap
        clf_tmp = LogisticRegression(max_iter=500)
        clf_tmp.fit(X_b.reshape(-1, 1), y_b)
        y_prob_b = clf_tmp.predict_proba(X_b.reshape(-1, 1))[:, 1]
        fpr_b, tpr_b, _ = roc_curve(y_b, y_prob_b)
        aucs.append(auc(fpr_b, tpr_b))

        if (i + 1) % 200 == 0:
            print(f"  Bootstrap {i+1}/{n_bootstrap} …")

    thresholds = np.array(thresholds)
    alpha_ci   = (1 - ci) / 2

    return {
        "threshold_median" : float(np.median(thresholds)),
        "ci_lower"         : float(np.quantile(thresholds, alpha_ci)),
        "ci_upper"         : float(np.quantile(thresholds, 1 - alpha_ci)),
        "thresholds"       : thresholds,
        "auc_median"       : float(np.median(aucs)),
        "n_valid"          : len(thresholds),
    }


print(f"Lancement bootstrap ({N_BOOTSTRAP} itérations) …")
boot_results = bootstrap_threshold(all_lots_df)

print("\n" + "═" * 60)
print(f"  SEUIL D'ALERTE ROBUSTE (IC {95}%)")
print(f"  Seuil médian      : {boot_results['threshold_median']:.4f}")
print(f"  Borne inférieure  : {boot_results['ci_lower']:.4f}")
print(f"  Borne supérieure  : {boot_results['ci_upper']:.4f}")
print(f"  AUC médiane       : {boot_results['auc_median']:.4f}")
print(f"  Itérations valides: {boot_results['n_valid']}/{N_BOOTSTRAP}")
print("═" * 60)

In [ ]:
# ── 4.3  Visualisation complète du bootstrap ──────────────────────────────────

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

thr_med   = boot_results['threshold_median']
thr_lo    = boot_results['ci_lower']
thr_hi    = boot_results['ci_upper']
thresholds = boot_results['thresholds']

# ── Panel A : distribution des seuils bootstrap ──────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.hist(thresholds, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
ax_a.axvline(thr_med, color='crimson',  linestyle='-',  lw=2, label=f"Médiane = {thr_med:.3f}")
ax_a.axvline(thr_lo,  color='darkorange', linestyle='--', lw=1.5, label=f"IC 95% [{thr_lo:.3f}, {thr_hi:.3f}]")
ax_a.axvline(thr_hi,  color='darkorange', linestyle='--', lw=1.5)
ax_a.fill_betweenx([0, ax_a.get_ylim()[1] if ax_a.get_ylim()[1] > 0 else 10],
                   thr_lo, thr_hi, color='orange', alpha=0.15)
ax_a.set_xlabel("Seuil d'entropie", fontsize=10)
ax_a.set_ylabel("Fréquence", fontsize=10)
ax_a.set_title(f"A — Distribution des seuils bootstrap (n={boot_results['n_valid']})", fontsize=11)
ax_a.legend(fontsize=9)
ax_a.grid(True, alpha=0.2)

# ── Panel B : courbe ROC sur le jeu complet ───────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
X_all = all_lots_df['entropy'].values
y_all = all_lots_df['label_lot'].values

clf_final = LogisticRegression(max_iter=500)
clf_final.fit(X_all.reshape(-1, 1), y_all)
y_proba_all = clf_final.predict_proba(X_all.reshape(-1, 1))[:, 1]

fpr_all, tpr_all, thr_roc = roc_curve(y_all, y_proba_all)
roc_auc_all = auc(fpr_all, tpr_all)

# Point correspondant au seuil médian bootstrap
w_f, b_f  = clf_final.coef_[0][0], clf_final.intercept_[0]
z_med     = w_f * thr_med + b_f
proba_med = expit(z_med)
idx_med   = np.argmin(np.abs(thr_roc - proba_med))

ax_b.plot(fpr_all, tpr_all, lw=2, color='steelblue',
          label=f"ROC (AUC = {roc_auc_all:.3f})")
ax_b.plot([0, 1], [0, 1], 'k--', lw=1, label="Aléatoire")
ax_b.scatter(fpr_all[idx_med], tpr_all[idx_med],
             color='crimson', zorder=5, s=60,
             label=f"Seuil médian bootstrap")
ax_b.set_xlabel("FPR", fontsize=10)
ax_b.set_ylabel("TPR", fontsize=10)
ax_b.set_title("B — Courbe ROC (jeu complet)", fontsize=11)
ax_b.legend(fontsize=9)
ax_b.grid(True, alpha=0.2)

# ── Panel C : entropie par lot avec seuil et IC ───────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
colors = all_lots_df['label_lot'].map({0: 'tab:green', 1: 'tab:red'})
ax_c.scatter(
    all_lots_df['score_perturbation'],
    all_lots_df['entropy'],
    c=colors, alpha=0.5, s=18, edgecolors='none'
)
ax_c.axhline(thr_med, color='crimson',    linestyle='-',  lw=2,   label=f"Seuil médian = {thr_med:.3f}")
ax_c.axhline(thr_lo,  color='darkorange', linestyle='--', lw=1.5, label=f"IC 95% : [{thr_lo:.3f}, {thr_hi:.3f}]")
ax_c.axhline(thr_hi,  color='darkorange', linestyle='--', lw=1.5)
ax_c.fill_between(
    [all_lots_df['score_perturbation'].min(), all_lots_df['score_perturbation'].max()],
    thr_lo, thr_hi, color='orange', alpha=0.15
)
# Légende couleur
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:green',
           markersize=8, label='Label 0 (propre)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:red',
           markersize=8, label='Label 1 (perturbé)'),
]
ax_c.legend(handles=legend_elements + ax_c.get_legend_handles_labels()[0][:2], fontsize=8)
ax_c.set_xlabel("Score de perturbation", fontsize=10)
ax_c.set_ylabel("Entropie hybride", fontsize=10)
ax_c.set_title("C — Entropie hybride vs perturbation", fontsize=11)
ax_c.grid(True, alpha=0.2)

# ── Panel D : stabilité du seuil (convergence avec n_bootstrap) ───────────────
ax_d = fig.add_subplot(gs[1, 1])
cumulative_median = np.array([
    np.median(thresholds[:k+1]) for k in range(len(thresholds))
])
ax_d.plot(range(1, len(thresholds)+1), cumulative_median,
          color='steelblue', lw=1.5, label="Médiane cumulative")
ax_d.axhline(thr_med, color='crimson', linestyle='--', lw=1.5,
             label=f"Valeur finale = {thr_med:.3f}")
ax_d.set_xlabel("Nombre d'itérations bootstrap", fontsize=10)
ax_d.set_ylabel("Seuil médian cumulatif", fontsize=10)
ax_d.set_title("D — Convergence du seuil bootstrap", fontsize=11)
ax_d.legend(fontsize=9)
ax_d.grid(True, alpha=0.2)

fig.suptitle(
    f"Seuil d'alerte robuste — config : batch={BEST_BATCH}, clustering={BEST_CLUSTERING}, "
    f"α_finance={best_alpha_finance:.2f}",
    fontsize=13, y=1.01
)

plt.savefig("seuil_bootstrap_final.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée → seuil_bootstrap_final.png")

In [ ]:
# ── 4.4  Résumé exécutif ──────────────────────────────────────────────────────

summary = {
    "Meilleure config" : {
        "batch_size"           : BEST_BATCH,
        "clustering"           : BEST_CLUSTERING,
    },
    "Alpha optimal" : {
        "finance (Test 1)"     : best_alpha_finance,
        "textuel (Test 2)"     : best_alpha_text,
    },
    "Seuil d'alerte" : {
        "médiane bootstrap"    : round(boot_results['threshold_median'], 4),
        "IC 95% inférieur"     : round(boot_results['ci_lower'], 4),
        "IC 95% supérieur"     : round(boot_results['ci_upper'], 4),
        "largeur IC"           : round(boot_results['ci_upper'] - boot_results['ci_lower'], 4),
    },
    "Performance" : {
        "AUC médiane bootstrap": round(boot_results['auc_median'], 4),
        "AUC globale (LogReg)" : round(roc_auc_all, 4),
    },
}

print("\n" + "═" * 60)
print("  RÉSUMÉ EXÉCUTIF")
print("═" * 60)
for section, vals in summary.items():
    print(f"\n  ▸ {section}")
    for k, v in vals.items():
        print(f"      {k:<30}: {v}")
print("═" * 60)

# Sauvegarde JSON
with open("resume_final.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("\nRésumé sauvegardé → resume_final.json")